# P41 — Adam: un método de optimización estocástica

## 1. Título y paper

**Paper:** *Adam: A Method for Stochastic Optimization*  
**Autoría:** Diederik P. Kingma, Jimmy Ba  
**Año y venue:** 2014 · arXiv:1412.6980 · ICLR 2015  
**Nivel:** L2 · **Motor:** `adam`  
**Ficha completa:** [`P41_adam`](../../papers/foundational/P41_adam/README.md)

**Hito:** Un paso de aprendizaje por dimensión, adaptado a la escala de su propio gradiente. Es el optimizador por defecto de casi todo lo que vino después.

- [arXiv:1412.6980](https://arxiv.org/abs/1412.6980)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: SGD usa la misma tasa de aprendizaje en todas las direcciones. En un problema mal condicionado, o oscila en las direcciones de mucha curvatura o se arrastra en las de poca.
2. Ejecutar una implementación mínima de la propuesta: Mantener medias móviles del gradiente (primer momento) y de su cuadrado (segundo momento), con corrección de sesgo, y normalizar el paso de cada coordenada por su magnitud típica.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02


## 4. Intuición

Bajar un valle largo y estrecho: en la dirección estrecha, un paso normal te hace rebotar de pared a pared; en la larga, ese mismo paso no avanza nada. Adam mide cuánto se mueve cada dirección y ajusta su paso por separado.


## 5. Concepto mínimo

```text
m_t = β₁·m_{t−1} + (1−β₁)·g          primer momento (dirección)
v_t = β₂·v_{t−1} + (1−β₂)·g²         segundo momento (escala)
m̂ = m_t/(1−β₁ᵗ),  v̂ = v_t/(1−β₂ᵗ)   corrección de sesgo
θ ← θ − η · m̂ / (√v̂ + ε)
```

El paso efectivo de cada coordenada es ~η, independientemente de la escala de su gradiente.


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('adam', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. Con L = x² + 100y², ¿qué gradiente es mayor: el de x o el de y?
2. ¿Qué le pasa a SGD con una tasa que sirva para x?
3. ¿Y a Adam?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('adam', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('adam', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

SGD se queda con una pérdida enorme: con una tasa que sirva para la dirección plana, **oscila** en la empinada sin converger. Adam llega a ~1e-8 porque normaliza cada coordenada por su propia escala de gradiente.


## 10. Comentario pedagógico

Adam es el optimizador por defecto de casi todo lo que estudias en este eje. Pero «por defecto» no es «siempre mejor»: hay trabajos que reportan mejor generalización con SGD con momento bien ajustado, sobre todo en visión.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar el decaimiento de pesos de SGD tal cual dentro de Adam.


In [ ]:
print('En SGD, weight decay equivale a sumar lambda*theta al gradiente.')
print('En Adam ese termino se divide tambien por sqrt(v): la regularizacion')
print('acaba siendo distinta por coordenada, que NO es lo que se queria.')
print('AdamW (2017) lo corrige desacoplandolo del paso adaptativo.')

## 12. Corrección

Los tres componentes y qué aporta cada uno:


In [ ]:
componentes = {'primer momento': 'suaviza la direccion, como el momento clasico',
               'segundo momento': 'normaliza por la escala tipica de cada coordenada',
               'correccion de sesgo': 'evita pasos diminutos en las primeras iteraciones'}
show(componentes)

## 13. Desafío guiado

Sube el número de condición del problema a 10 000 y comprueba si Adam sigue convergiendo.


In [ ]:
r = run_paper_lab('adam', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena la misma red con SGD, SGD+momento, Adam y AdamW sobre un conjunto pequeño. Compara curvas de entrenamiento **y** de validación: el mejor entrenamiento no siempre generaliza mejor.


## 15. Evidencia de aprendizaje

Guarda la comparación en el problema mal condicionado y tu explicación de por qué el decaimiento de pesos necesita tratamiento aparte.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P41_adam/README.md) · evaluación formal: [`assessments/papers/P41_adam.md`](../../assessments/papers/P41_adam.md)


## 16. Cierre

Ya se entrena estable y rápido. Toca la pregunta incómoda: ¿es robusto lo que se ha entrenado?


## 17. Conexión con el siguiente hito

- entrenamiento de todos los modelos posteriores

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
